# Ploemeur temporal example

This notebook mirrors the teaching structure of the single-date `ploemeur` example, but here the field case is transient rather than single-date.

The goal is to understand how the observations are distributed in time, which temporal calibration mode is being used, and how to read the compact figure set written for each calibrated case.

Set `expert_mode = False` if you want a lighter notebook. Keep it at `True` to inspect the enlarged expert diagnostics at the end.


## Notebook roadmap

This notebook follows nine short steps:

1. review dataset and workflow settings;
2. inspect calibration parameters;
3. inspect observed time series;
4. run the temporal workflow;
5. read the compact figures case by case;
6. read calibrated parameter tables;
7. compare transient and single-date parameter distributions;
8. inspect the objective landscape and the MH solutions;
9. inspect expert diagnostics when needed.


In [ ]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, Markdown, display
from IPython.utils.capture import capture_output

from pyages.calibration.exploration.systematic import SystematicSampling
from pyages.concentrations import Concentrations
from pyages.lpm import build_lpm
from pyages.workflows.temporal import run_temporal
from pyages.reporting.plots import (
    plot_objective_solution_map,
    plot_objective_summary,
    plot_observations_overview,
    plot_parameter_distribution_comparison,
    plot_temporal_fit_comparison,
)
from pyages.workflows.single_date import run_single_date
from pyages.workflows.single_date.config import load_params
from pyages.workflows.single_date.paths import dataset_results_directory

ROOT = Path.cwd().resolve()
while not (ROOT / "pyages").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "pyages").exists():
    raise RuntimeError(
        "Run this notebook from the repository root or one of its subdirectories."
    )

EXAMPLE_DIR = ROOT / "examples" / "natural" / "ploemeur_temporal"


def resolve_path(path_str: str) -> Path:
    path = Path(path_str)
    return path if path.is_absolute() else ROOT / path


def read_tsv(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, sep="	")
    return frame.loc[:, ~frame.columns.str.startswith("Unnamed")]


def read_stats(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep="	", index_col=0)


def ensure_single_date_results(params_path: Path, lpm_name: str) -> Path:
    base_cfg = yaml.safe_load(params_path.read_text(encoding="utf-8"))
    base_model_name = base_cfg["lpm"]["model_name"]
    if lpm_name == base_model_name:
        params = load_params(ROOT, params_path)
        expected_dir = Path(dataset_results_directory(params.dataset_name))
        posterior_path = (
            expected_dir / "Metropolis_Hastings" / "lpm_dist_calibrated.txt"
        )
        if not posterior_path.exists():
            with capture_output():
                expected_dir = Path(
                    run_single_date(params_path, force_inline=True)
                )
        return expected_dir

    compare_root = EXAMPLE_DIR / "_single_date_compare"
    compare_root.mkdir(parents=True, exist_ok=True)
    data_dir = compare_root / "data"
    data_dir.mkdir(parents=True, exist_ok=True)

    source_data_dir = resolve_path(base_cfg["dataset"]["data_dir"])
    source_dataset = source_data_dir / base_cfg["dataset"]["name"]
    dataset_stem = Path(base_cfg["dataset"]["name"]).stem
    generated_dataset_name = f"{dataset_stem}_{lpm_name}.txt"
    generated_dataset = data_dir / generated_dataset_name
    if not generated_dataset.exists():
        shutil.copyfile(source_dataset, generated_dataset)

    compare_cfg = yaml.safe_load(params_path.read_text(encoding="utf-8"))
    compare_cfg["dataset"]["name"] = generated_dataset_name
    compare_cfg["dataset"]["data_dir"] = str(data_dir)
    compare_cfg["dataset"]["label"] = f"Ploemeur single-date example ({lpm_name})"
    compare_cfg["lpm"]["model_name"] = lpm_name
    compare_params_path = compare_root / f"exemple_ploemeur_{lpm_name}.yaml"
    compare_params_path.write_text(
        yaml.safe_dump(compare_cfg, sort_keys=False), encoding="utf-8"
    )

    expected_dir = Path(dataset_results_directory(generated_dataset_name))
    posterior_path = expected_dir / "Metropolis_Hastings" / "lpm_dist_calibrated.txt"
    if not posterior_path.exists():
        with capture_output():
            expected_dir = Path(
                run_single_date(compare_params_path, force_inline=True)
            )
    return expected_dir


def case_directories(results_root: Path, mode: str):
    if mode == "span":
        return [results_root]
    return sorted(
        path
        for path in results_root.iterdir()
        if path.is_dir() and path.name.startswith("date_")
    )


def case_title(case_dir: Path) -> str:
    if case_dir.name == "span_full":
        return "Full time span"
    label = case_dir.name[5:] if case_dir.name.startswith("date_") else case_dir.name
    return f"Sampling date {label.replace('_', '.')}"


def case_concentrations(
    case_dir: Path, source_data: Concentrations, mode: str
) -> Concentrations:
    if mode == "span":
        frame = source_data.frame.copy()
    else:
        label = (
            case_dir.name[5:] if case_dir.name.startswith("date_") else case_dir.name
        )
        date_value = float(label.replace("_", "."))
        mask = np.isclose(source_data.frame["date"].astype(float), date_value)
        frame = source_data.frame.loc[mask].copy()
        if frame.empty:
            raise ValueError(f"No observations found for {case_dir.name}.")
    return Concentrations.from_dataframe(frame)


def objective_sampling_nmodels(param_names: list[str], base_nmodels: int) -> int:
    nparams = max(len(param_names), 1)
    min_levels = 25 if nparams == 1 else 8 if nparams == 2 else 5 if nparams == 3 else 4
    return max(int(base_nmodels), min_levels**nparams)


def build_objective_artifacts(
    case_dir: Path,
    lpm_name: str,
    case_data: Concentrations,
    base_nmodels: int,
    lpm_directory: Path,
) -> dict:
    notebook_dir = case_dir / "notebook_views"
    notebook_dir.mkdir(parents=True, exist_ok=True)

    posterior_path = case_dir / lpm_name / "lpm_dist_calibrated.txt"
    posterior_frame = read_tsv(posterior_path)
    param_names = build_lpm(
        lpm_name, directory_lpm=str(lpm_directory)
    ).get_param_names()
    objective_nmodels = objective_sampling_nmodels(param_names, base_nmodels)

    sampling = SystematicSampling(
        lpm_name,
        case_data.observation_tracer_names(),
        date=case_data.frame["date"].tolist(),
        observations=case_data,
        sample_count=objective_nmodels,
        display_options=None,
        explore_objective=True,
        explore_reachable=False,
    )
    sampling.compute_concentrations()
    sampling.objective_function_build()
    objective_frame = sampling.objective_function_frame()

    objective_grid_path = notebook_dir / f"objective_function_grid_{lpm_name}.txt"
    summary_path = notebook_dir / f"03_mh_objective_summary_{lpm_name}.png"
    solution_map_path = notebook_dir / f"04_mh_objective_solutions_{lpm_name}.png"

    objective_frame.to_csv(objective_grid_path, sep="	", index=False)

    title_prefix = f"{case_title(case_dir)} - {lpm_name}"
    fig = plot_objective_summary(
        objective_frame=objective_frame,
        posterior_results={"Metropolis_Hastings": posterior_frame},
        param_names=param_names,
        filename=summary_path,
        title=f"{title_prefix}: objective landscape and MH solutions",
    )
    plt.close(fig)

    fig = plot_objective_solution_map(
        objective_frame=objective_frame,
        posterior_frame=posterior_frame,
        param_names=param_names,
        filename=solution_map_path,
        title=f"{title_prefix}: posterior positioning on the objective landscape",
    )
    plt.close(fig)

    return {
        "objective_grid": objective_grid_path,
        "summary": summary_path,
        "solutions": solution_map_path,
        "param_names": param_names,
        "nmodels": objective_nmodels,
    }


def ensure_objective_artifacts(
    cache: dict,
    case_dir: Path,
    lpm_name: str,
    case_data: Concentrations,
    base_nmodels: int,
    lpm_directory: Path,
) -> dict:
    key = (str(case_dir), lpm_name)
    if key not in cache:
        cache[key] = build_objective_artifacts(
            case_dir=case_dir,
            lpm_name=lpm_name,
            case_data=case_data,
            base_nmodels=base_nmodels,
            lpm_directory=lpm_directory,
        )
    return cache[key]


expert_mode = True
focus_lpm = None  # Example: 'ig'
focus_case = None  # Example: 'date_2010'
case_display_limit = 3  # successive mode only

## 1. Dataset and workflow settings

Before running anything, review the field dataset, its temporal coverage, the tracer list, and the workflow mode selected in the YAML file.

In [ ]:
params_path = EXAMPLE_DIR / "ploemeur_temporal.yaml"
params = yaml.safe_load(params_path.read_text(encoding="utf-8"))

dataset_path = resolve_path(params["dataset"]["file"])
observations = Concentrations.from_file(dataset_path)
error_rel = params["dataset"].get("error_rel")
if error_rel is not None and (observations.frame["error"] == 0).any():
    observations.set_relative_errors(float(error_rel))

lpm_list = params.get("lpm_models", {}).get("list") or [
    "exp_shifted",
    "ig",
    "ig_shifted",
]
display_lpms = lpm_list if focus_lpm is None else [focus_lpm]

date_values = sorted(float(date) for date in observations.frame["date"].unique())
single_date_params_path = (
    ROOT / "examples" / "natural" / "ploemeur" / "exemple_ploemeur.yaml"
)
single_date_cfg = yaml.safe_load(single_date_params_path.read_text(encoding="utf-8"))
single_date_dataset_path = resolve_path(
    str(
        Path(single_date_cfg["dataset"]["data_dir"])
        / single_date_cfg["dataset"]["name"]
    )
)
single_date_data = Concentrations.from_file(single_date_dataset_path)
single_date_reference_dates = sorted(
    float(date) for date in single_date_data.frame["date"].unique()
)
if len(single_date_reference_dates) != 1:
    raise ValueError(
        "The single-date reference dataset must contain exactly one sampling date."
    )
single_date_reference_date = single_date_reference_dates[0]
single_date_temporal_date = min(
    date_values, key=lambda date: abs(date - single_date_reference_date)
)
tracer_names = ", ".join(sorted(name.upper() for name in observations.unique_tracer_names()))
case_rows = [
    {"parameter": "dataset file", "value": params["dataset"]["file"]},
    {"parameter": "data path", "value": str(dataset_path)},
    {"parameter": "time span", "value": f"{date_values[0]} to {date_values[-1]}"},
    {"parameter": "number of dates", "value": len(date_values)},
    {
        "parameter": "single-date reference",
        "value": f"{single_date_reference_date:.1f} matched to {single_date_temporal_date:.3f}",
    },
    {"parameter": "tracers", "value": tracer_names},
    {"parameter": "workflow mode", "value": params["workflow"]["mode"]},
    {"parameter": "LPMs", "value": ", ".join(lpm_list)},
    {"parameter": "displayed LPMs", "value": ", ".join(display_lpms)},
    {"parameter": "expert mode", "value": expert_mode},
]

display(pd.DataFrame(case_rows))
display(Markdown(f"Configuration file: `{params_path}`"))

## 2. Calibration parameters

In the transient case, the main structural choice is the workflow mode:

- `span`: one calibration uses the full observation chronicle;
- `successive`: one calibration is run for each observation date.

This choice changes both the interpretation and the result folder layout.

In [ ]:
workflow_mode = params["workflow"]["mode"]
calibration_rows = [
    {
        "parameter": "relative error fallback",
        "value": params["dataset"].get("error_rel"),
    },
    {
        "parameter": "systematic sampling resolution",
        "value": params["calibration"]["explo_res"],
    },
    {"parameter": "MH steps", "value": params["calibration"]["mh_nsteps"]},
    {"parameter": "burn-in fraction", "value": params["calibration"]["burn_in"]},
    {"parameter": "thinning interval", "value": params["calibration"]["nskip"]},
    {"parameter": "seed enabled", "value": params["calibration"]["seed_enabled"]},
    {"parameter": "seed", "value": params["calibration"]["seed"]},
    {"parameter": "temporal figures", "value": params["figures"]["temporal"]},
    {"parameter": "distribution figures", "value": params["figures"]["distributions"]},
    {
        "parameter": "2D concentration figures",
        "value": params["figures"]["concentrations_2d"],
    },
]

if workflow_mode == "successive":
    shown_cases = focus_case or f"first {case_display_limit} calibrated dates"
    calibration_rows.append({"parameter": "displayed cases", "value": shown_cases})

display(pd.DataFrame(calibration_rows))

## 3. Observed concentrations

Start with the observations only. This is the fastest way to see the temporal coverage, tracer spread, and the years that will be easiest or hardest to fit.

In [ ]:
display(observations.frame.head(12))
plot_observations_overview(
    observations,
    title="Observed concentrations before calibration",
    highlight_dates=[single_date_reference_date],
    highlight_label="Single-date interpretation",
)
display(
    Markdown(
        f"The red point marks the observation reused by the single-date example ({single_date_reference_date:.1f}; matched to {single_date_temporal_date:.3f} in the temporal file)."
    )
)

## 4. Run the temporal workflow

The launcher writes one observation overview per calibrated case and, for each LPM, a temporal fit summary plus a compact parameter summary.

In [ ]:
with capture_output():
    results_root = Path(run_temporal(params_path))

display(Markdown(f"**Results directory:** `{results_root}`"))
results_root

## 5. Beginner view: read the compact outputs first

Read each calibrated case in this order:

1. observation overview;
2. temporal fit summary;
3. parameter summary.

In [ ]:
all_case_dirs = case_directories(results_root, workflow_mode)

if focus_case is not None:
    case_dirs = [case_dir for case_dir in all_case_dirs if case_dir.name == focus_case]
elif workflow_mode == "successive":
    case_dirs = all_case_dirs[:case_display_limit]
else:
    case_dirs = all_case_dirs

if not case_dirs:
    raise ValueError(
        "No result case matched focus_case. Inspect results_root and update focus_case."
    )

if (
    workflow_mode == "successive"
    and focus_case is None
    and len(all_case_dirs) > len(case_dirs)
):
    display(
        Markdown(
            f"Showing the first {len(case_dirs)} calibrated dates out of {len(all_case_dirs)}. "
            "Set `focus_case` to inspect another date-specific folder."
        )
    )

for case_dir in case_dirs:
    case_data = case_concentrations(case_dir, observations, workflow_mode)
    display(Markdown(f"## {case_title(case_dir)}"))
    overview = case_dir / "00_observations_overview.png"
    if overview.exists():
        display(Markdown("### Observation overview"))
        notebook_dir = case_dir / "notebook_views"
        notebook_dir.mkdir(parents=True, exist_ok=True)
        overview_highlighted = notebook_dir / "00_observations_overview_single_date.png"
        fig = plot_observations_overview(
            case_data,
            filename=overview_highlighted,
            title="Observed concentrations before calibration",
            highlight_dates=[single_date_reference_date],
            highlight_label="Single-date interpretation",
        )
        plt.close(fig)
        display(Image(filename=str(overview_highlighted), width=950))
        display(
            Markdown(
                "Start here. This figure shows the raw temporal spread, the uncertainty bars, and the years the model has to explain. The red point is the observation reused by the single-date reference example."
            )
        )

    for lpm_name in display_lpms:
        lpm_dir = case_dir / lpm_name
        fit_path = lpm_dir / "Metropolis_Hastings" / "concentration_times.png"
        param_path = lpm_dir / "parameter_summary.png"
        display(Markdown(f"### {lpm_name}"))
        if fit_path.exists():
            display(Markdown("#### Temporal fit summary"))
            display(Image(filename=str(fit_path), width=950))
            display(
                Markdown(
                    "Check whether the observed points stay inside the posterior bands and whether the same years are missed repeatedly."
                )
            )
        if param_path.exists():
            display(Markdown("#### Parameter summary"))
            display(Image(filename=str(param_path), width=900))
            display(
                Markdown(
                    "A tight posterior means the transient dataset constrains the parameter well. A broad posterior means several histories remain plausible."
                )
            )

## 6. Read the calibrated parameter tables

After the figures, the next useful object is the per-case, per-LPM statistics table. Read each case independently before comparing across dates or across LPMs.

In [ ]:
for case_dir in case_dirs:
    display(Markdown(f"## {case_title(case_dir)}"))
    for lpm_name in display_lpms:
        stats_path = case_dir / lpm_name / "lpm_stats_calibrated.txt"
        if not stats_path.exists():
            continue
        stats = read_stats(stats_path)
        display(Markdown(f"### {lpm_name}"))
        display(stats.loc[["count", "mean", "std"]])

## 7. Compare transient and single-date solutions

This section groups the direct comparisons between the transient calibration and the single-date `F09_2010` example for both shifted models: `exp_shifted` and `ig_shifted`.

Read the temporal-fit overlay first, then the parameter distributions, to see whether the time information mainly narrows the uncertainty or also shifts the preferred parameter range.

In [ ]:
single_date_params_path = (
    ROOT / "examples" / "natural" / "ploemeur" / "exemple_ploemeur.yaml"
)
temporal_lpm_directory = resolve_path(
    params.get("lpm_models", {}).get("directory") or "data_core/data_lpm"
)
default_lpm_number = int(params["calibration"]["lpm_number"])
if default_lpm_number <= 0:
    default_lpm_number = max(
        min(int(params["calibration"]["mh_nsteps"] / 50), 5000), 10
    )
comparison_param_candidates = ["mu", "sigma", "shift"]
comparison_param_labels = {"mu": r"$\mu$", "sigma": r"$\sigma$", "shift": r"$t_0$"}
comparison_param_density_labels = {"mu": r"\mu", "sigma": r"\sigma", "shift": r"t_0"}

for case_dir in case_dirs:
    case_data = case_concentrations(case_dir, observations, workflow_mode)
    display(Markdown(f"## {case_title(case_dir)}"))
    for lpm_name in display_lpms:
        if lpm_name not in {"exp_shifted", "ig_shifted"}:
            continue
        transient_path = case_dir / lpm_name / "lpm_dist_calibrated.txt"
        if not transient_path.exists():
            continue

        single_date_results_dir = ensure_single_date_results(
            single_date_params_path, lpm_name
        )
        single_date_posterior = read_tsv(
            single_date_results_dir / "Metropolis_Hastings" / "lpm_dist_calibrated.txt"
        )
        transient_posterior = read_tsv(transient_path)
        notebook_dir = case_dir / "notebook_views"
        notebook_dir.mkdir(parents=True, exist_ok=True)

        temporal_comparison_path = (
            notebook_dir / f"02_temporal_fit_summary_comparison_{lpm_name}.png"
        )
        fig = plot_temporal_fit_comparison(
            craw=case_data,
            posterior_frames={
                "Transient posterior": transient_posterior,
                "Single-date posterior": single_date_posterior,
            },
            lpm_name=lpm_name,
            lpm_directory=temporal_lpm_directory,
            highlight_dates=[single_date_reference_date],
            highlight_label="Single-date observation",
            lpm_number=default_lpm_number,
            filename=temporal_comparison_path,
            title=None,
        )
        plt.close(fig)

        param_names = [
            name
            for name in comparison_param_candidates
            if name in transient_posterior.columns
            and name in single_date_posterior.columns
        ]
        if not param_names:
            continue
        parameter_comparison_path = (
            notebook_dir
            / f"05_transient_vs_single_date_{lpm_name}_parameter_distributions.png"
        )
        fig = plot_parameter_distribution_comparison(
            distributions={
                "Transient posterior": transient_posterior,
                "Single-date posterior": single_date_posterior,
            },
            param_names=param_names,
            param_labels=comparison_param_labels,
            param_density_labels=comparison_param_density_labels,
            filename=parameter_comparison_path,
            title=None,
        )
        plt.close(fig)

        display(Markdown(f"### {lpm_name}"))
        display(
            Markdown(
                f"**Single-date reference directory:** `{single_date_results_dir}`"
            )
        )
        display(Markdown("#### Temporal fit comparison"))
        display(Image(filename=str(temporal_comparison_path), width=980))
        display(
            Markdown(
                "A narrower transient band means the time series constrains the model more strongly; a shifted median means it prefers a different age structure."
            )
        )
        display(Markdown("#### Parameter distribution comparison"))
        display(Image(filename=str(parameter_comparison_path), width=980))
        display(
            Markdown(
                "If the transient curves are narrower than the single-date curves, the time series adds constraint. "
                "If the peaks move, the transient information changes the preferred age structure rather than only reducing uncertainty."
            )
        )

## 8. Objective landscape and MH solutions

This extra step mirrors the single-date example. It rebuilds a coarse prior objective grid for each displayed case and overlays the Metropolis-Hastings posterior solutions on top of it.

Use it to check whether the accepted solutions really occupy the lowest-objective zone or whether the chain still spreads into broader parameter trade-offs.


In [ ]:
objective_artifacts = {}
objective_base_nmodels = int(params["calibration"]["explo_res"])
objective_lpm_directory = resolve_path(
    params.get("lpm_models", {}).get("directory") or "data_core/data_lpm"
)

for case_dir in case_dirs:
    case_data = case_concentrations(case_dir, observations, workflow_mode)
    display(Markdown(f"## {case_title(case_dir)}"))
    for lpm_name in display_lpms:
        posterior_path = case_dir / lpm_name / "lpm_dist_calibrated.txt"
        if not posterior_path.exists():
            continue
        artifacts = ensure_objective_artifacts(
            objective_artifacts,
            case_dir=case_dir,
            lpm_name=lpm_name,
            case_data=case_data,
            base_nmodels=objective_base_nmodels,
            lpm_directory=objective_lpm_directory,
        )
        display(Markdown(f"### {lpm_name}"))
        display(Image(filename=str(artifacts["summary"]), width=1200))
        display(
            Markdown(
                f"This notebook view rebuilds a coarse prior objective grid with `{artifacts['nmodels']}` forward models. "
                "The colored background comes from the prior grid, the blue cloud from the MH posterior, "
                "the white star from the best prior grid point, and the blue star from the best MH solution."
            )
        )

## 9. Expert mode

Set `expert_mode = True` when you want the enlarged posterior-on-objective view, the raw posterior samples, the full statistics tables, or the optional 2D concentration diagnostics.


In [ ]:
if "objective_artifacts" not in globals():
    objective_artifacts = {}

if expert_mode:
    objective_base_nmodels = int(params["calibration"]["explo_res"])
    objective_lpm_directory = resolve_path(
        params.get("lpm_models", {}).get("directory") or "data_core/data_lpm"
    )

    for case_dir in case_dirs:
        case_data = case_concentrations(case_dir, observations, workflow_mode)
        display(Markdown(f"## {case_title(case_dir)}"))
        for lpm_name in display_lpms:
            lpm_dir = case_dir / lpm_name
            dist_path = lpm_dir / "lpm_dist_calibrated.txt"
            stats_path = lpm_dir / "lpm_stats_calibrated.txt"
            display(Markdown(f"### {lpm_name}"))

            if dist_path.exists():
                artifacts = ensure_objective_artifacts(
                    objective_artifacts,
                    case_dir=case_dir,
                    lpm_name=lpm_name,
                    case_data=case_data,
                    base_nmodels=objective_base_nmodels,
                    lpm_directory=objective_lpm_directory,
                )
                display(Markdown("#### Posterior solutions on the objective landscape"))
                display(Image(filename=str(artifacts["solutions"]), width=1220))
                display(
                    Markdown(
                        "Use this enlarged view to see whether the accepted MH samples stay packed inside the lowest-objective basin. "
                        "A compact low-objective cloud indicates a well-focused transient calibration; "
                        "a diffuse or multi-lobed cloud indicates that parameter trade-offs remain."
                    )
                )
                display(Markdown("#### First calibrated samples"))
                display(read_tsv(dist_path).head(10))

            if stats_path.exists():
                display(Markdown("#### Full statistics table"))
                display(read_stats(stats_path))

            pair_plots = sorted(lpm_dir.glob("concentrations2D_*.png"))
            if pair_plots:
                display(Markdown("#### Example 2D concentration diagnostic"))
                display(Image(filename=str(pair_plots[0]), width=1120))
                display(
                    Markdown(
                        "This projection compares two tracer concentrations at once. "
                        "If the observation sits outside the posterior cloud, the LPM struggles to explain those tracer constraints jointly."
                    )
                )

    display(Markdown("### Result folders"))
    for path in sorted(results_root.iterdir()):
        print(path.name)
else:
    print(
        "Set expert_mode = True and re-run this cell to display enlarged objective diagnostics, raw posterior samples, and optional 2D concentration plots."
    )